# Task 1: LoRA Fine-Tuning on Qwen2.5-0.5B-Instruct

This notebook fine-tunes **Qwen/Qwen2.5-0.5B-Instruct** with **LoRA (Low-Rank Adaptation)**
on a natural-language → structured JSON dataset for a calendar assistant.

**Run this notebook on Google Colab with a GPU runtime (T4 or better).**

In [ ]:
# ── Cell 2: Dependency Installation ──────────────────────────────────────────
# 1. Uninstall torchao to prevent tuner registry / dtype conflicts with PEFT in Colab
!pip uninstall -y torchao -q

# 2. Install mutually compatible, tested dependency set
!pip install --upgrade "transformers>=4.48.0" "peft>=0.14.0" "accelerate>=1.3.0" "datasets>=3.2.0" "sentencepiece>=0.2.0" -q

print("Environment setup complete! Core dependencies installed cleanly.")

## Setup and Configuration

### LoRA Hyperparameter Choices & Engineering Rationale

| Hyperparameter | Value | Justification |
|---|---|---|
| `r` (rank) | 16 | Balances expressiveness vs. parameter overhead. Rank-16 adds ~1.2% extra parameters (~6M params). |
| `lora_alpha` | 32 | Scaling factor $\alpha / r = 32 / 16 = 2.0$. Normalizes adapter updates; maintains stable gradient norms. |
| `lora_dropout` | 0.1 | Light regularisation on low-rank paths; mitigates overfitting on ~700 examples. |
| `target_modules` | q, k, v, o proj | Attention projections carry sequence transformation capabilities for structured generation. |
| `learning_rate` | 2e-4 | Higher than full fine-tuning (2e-5) because base weights are frozen and adapters start near zero. |
| `warmup_steps` | 5 | Equivalent to 5% warmup over 96 total optimization steps ($503/16 \times 3 \approx 96$ steps). Prevents early gradient spikes. |
| `lr_scheduler` | cosine | Smooth decay without abrupt learning rate drops. |
| `batch_size` | 4 | Fits comfortably in T4 16 GB VRAM with fp16. |
| `gradient_accumulation` | 4 | Effective batch size = $4 \times 4 = 16$; stabilizes mini-batch gradients. |
| `epochs` | 3 | Sufficient convergence for slot extraction while preventing memorization of training utterances. |

In [ ]:
# ── Cell 4: Reproducibility & Device Check ──────────────────────────────────
import os
import sys
import json
import random
import numpy as np
import torch

# Set seeds for full reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: Running on CPU. Training will be slow without a GPU (Runtime > Change runtime type > T4 GPU).")

In [ ]:
# ── Cell 5: Path Resolution & Environment Mounting ─────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_PATH = '/content/drive/MyDrive/tndrr-ai-ml-assignment/task-1-lora-finetuning'
except Exception as e:
    print(f"Drive mount skipped: {e}")
    REPO_PATH = '.'

# Fallback detection if repository was cloned or uploaded directly to /content
if not os.path.exists(REPO_PATH):
    candidates = [
        '/content/tndrr-ai-ml-assignment/task-1-lora-finetuning',
        '/content/task-1-lora-finetuning',
        './task-1-lora-finetuning',
        '.'
    ]
    for cand in candidates:
        if os.path.exists(os.path.join(cand, 'dataset')):
            REPO_PATH = cand
            break

DATASET_DIR = os.path.join(REPO_PATH, 'dataset')
RESULTS_DIR = os.path.join(REPO_PATH, 'results')
SRC_DIR = os.path.join(REPO_PATH, 'src')

os.makedirs(RESULTS_DIR, exist_ok=True)
if os.path.exists(SRC_DIR) and SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print(f"REPO_PATH:   {os.path.abspath(REPO_PATH)}")
print(f"DATASET_DIR: {os.path.abspath(DATASET_DIR)} (found: {os.path.exists(DATASET_DIR)})")
print(f"RESULTS_DIR: {os.path.abspath(RESULTS_DIR)}")

In [ ]:
# ── Cell 6: Dataset Loading & Formatting ────────────────────────────────────
from datasets import Dataset

INSTRUCTION = "Convert the following natural language to structured JSON."

def format_example(example):
    inp = example["input"]
    out_str = json.dumps(example["output"], ensure_ascii=False)
    return f"### Instruction:\n{INSTRUCTION}\n\n### Input:\n{inp}\n\n### Response:\n{out_str}"

# Load datasets from JSON
def load_json_file(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

train_examples = load_json_file(os.path.join(DATASET_DIR, 'train.json'))
val_examples   = load_json_file(os.path.join(DATASET_DIR, 'val.json'))
test_examples  = load_json_file(os.path.join(DATASET_DIR, 'test.json'))

print(f"Dataset Splits: Train={len(train_examples)} | Val={len(val_examples)} | Test={len(test_examples)}")
print("\nSample Formatted Example:")
print(format_example(train_examples[0]))

# Create HuggingFace datasets
train_hf = Dataset.from_list([{"text": format_example(ex)} for ex in train_examples])
val_hf   = Dataset.from_list([{"text": format_example(ex)} for ex in val_examples])
print("HuggingFace Datasets successfully created.")

## LoRA Configuration

### Why these settings?

**Rank (`r=16`)**: The rank controls how many extra parameters we add. For a weight matrix
W ∈ ℝ^(d×k), LoRA adds BA where B ∈ ℝ^(d×r), A ∈ ℝ^(r×k). With r=16, d=896, k=896
(Qwen2.5-0.5B hidden size), that is 16×896×2 = ~28K extra params per layer — a tiny
fraction of the full layer's 896² ≈ 800K.

**Alpha (`lora_alpha=32`)**: The effective learning rate scale is α/r = 32/16 = 2. This
ensures the LoRA updates are of a similar magnitude to typical full fine-tuning updates
without needing to change the base learning rate.

**Target modules**: Applying LoRA to `q_proj`, `k_proj`, `v_proj`, `o_proj` covers all
four linear projections in multi-head self-attention. These carry most of the structured
reasoning capability that needs task adaptation.

**Dropout (`lora_dropout=0.1`)**: Acts as regularisation on the low-rank paths only,
reducing the risk of overfitting to our ~700-example dataset.

In [ ]:
# ── Cell 8: Model & LoRA Initialization ─────────────────────────────────────
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

# Model dtype setup
model_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

# Load base model
print(f"Loading {MODEL_ID} in {model_dtype}...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=model_dtype,
    device_map="auto" if torch.cuda.is_available() else None,
)
base_model.config.use_cache = False  # required for training / gradient checkpointing

# LoRA configuration
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)

# Apply LoRA
model = get_peft_model(base_model, lora_config)
print("LoRA Adapter Applied Successfully:")
model.print_trainable_parameters()

In [ ]:
# ── Cell 9: Training with Native Hugging Face Trainer ───────────────────────
import inspect
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

OUTPUT_DIR = os.path.join(RESULTS_DIR, 'lora-qwen-calendar')
ADAPTER_DIR = os.path.join(RESULTS_DIR, 'lora-adapters')

# Tokenize datasets
def tokenize_fn(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding=False,
    )

train_tokenized = train_hf.map(tokenize_fn, batched=True, remove_columns=["text"])
val_tokenized   = val_hf.map(tokenize_fn,   batched=True, remove_columns=["text"])

# Inspect TrainingArguments constructor for universal version compatibility
ta_params = inspect.signature(TrainingArguments.__init__).parameters
eval_arg_name = "eval_strategy" if "eval_strategy" in ta_params else "evaluation_strategy"

args_dict = {
    "output_dir": OUTPUT_DIR,
    "per_device_train_batch_size": 4,
    "gradient_accumulation_steps": 4,
    "num_train_epochs": 3,
    "learning_rate": 2e-4,
    "fp16": torch.cuda.is_available(),
    "logging_steps": 10,
    eval_arg_name: "epoch",
    "save_strategy": "epoch",
    "warmup_steps": 5,  # exact equivalent of 5% warmup over 96 total optimization steps
    "lr_scheduler_type": "cosine",
    "load_best_model_at_end": True,
    "report_to": "none",
    "seed": SEED,
}

training_args = TrainingArguments(**args_dict)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
)

print("Starting LoRA fine-tuning...")
trainer.train()

# Save trained LoRA adapters
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"LoRA adapters saved successfully to: {ADAPTER_DIR}")

In [ ]:
# ── Cell 10: Evaluate Base Model on Test Set ────────────────────────────────
import re

def format_prompt(input_text):
    return f"### Instruction:\n{INSTRUCTION}\n\n### Input:\n{input_text}\n\n### Response:\n"

def extract_json_from_response(text: str) -> str:
    """Extract outermost JSON object from output."""
    start = text.find('{')
    end = text.rfind('}')
    if start != -1 and end != -1 and end > start:
        return text[start:end+1]
    return text.strip()

def run_inference(eval_model, eval_tokenizer, examples, max_new_tokens=128):
    eval_model.eval()
    predictions = []
    target_device = next(eval_model.parameters()).device
    for ex in examples:
        prompt = format_prompt(ex['input'])
        inputs = eval_tokenizer(prompt, return_tensors='pt').to(target_device)
        with torch.no_grad():
            output_ids = eval_model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=eval_tokenizer.eos_token_id,
            )
        new_tokens = output_ids[0][inputs['input_ids'].shape[1]:]
        response = eval_tokenizer.decode(new_tokens, skip_special_tokens=True)
        pred_json = extract_json_from_response(response)
        predictions.append({'input': ex['input'], 'gold': ex['output'], 'prediction': pred_json})
    return predictions

print("Loading fresh base model for baseline evaluation...")
base_model_eval = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=model_dtype,
    device_map="auto" if torch.cuda.is_available() else None,
)
base_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
base_tokenizer.pad_token = base_tokenizer.eos_token

print(f"Running base model inference on {len(test_examples)} held-out test examples...")
base_predictions = run_inference(base_model_eval, base_tokenizer, test_examples)

base_preds_path = os.path.join(RESULTS_DIR, 'predictions_base.json')
with open(base_preds_path, 'w', encoding="utf-8") as f:
    json.dump(base_predictions, f, indent=2)
print(f"Base predictions saved to: {base_preds_path}")

In [ ]:
# ── Cell 11: Evaluate Fine-Tuned Model on Test Set ──────────────────────────
from peft import PeftModel

print(f"Loading fine-tuned adapter from {ADAPTER_DIR}...")
ft_base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=model_dtype,
    device_map="auto" if torch.cuda.is_available() else None,
)
ft_model = PeftModel.from_pretrained(ft_base, ADAPTER_DIR)
ft_tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
ft_tokenizer.pad_token = ft_tokenizer.eos_token

print(f"Running fine-tuned model inference on {len(test_examples)} held-out test examples...")
ft_predictions = run_inference(ft_model, ft_tokenizer, test_examples)

ft_preds_path = os.path.join(RESULTS_DIR, 'predictions_finetuned.json')
with open(ft_preds_path, 'w', encoding="utf-8") as f:
    json.dump(ft_predictions, f, indent=2)
print(f"Fine-tuned predictions saved to: {ft_preds_path}")

In [ ]:
# ── Cell 12: Compute & Compare Evaluation Metrics ───────────────────────────
def is_valid_json(text: str) -> bool:
    try:
        json.loads(text)
        return True
    except:
        return False

def field_exact_match(pred_dict, gold_dict):
    result = {}
    for k, v in gold_dict.items():
        pred_v = pred_dict.get(k)
        if isinstance(v, str) and isinstance(pred_v, str):
            result[k] = v.strip().lower() == pred_v.strip().lower()
        else:
            result[k] = v == pred_v
    return result

def compute_metrics(predictions, ground_truth):
    n = len(predictions)
    n_valid = 0
    n_overall_exact = 0
    action_matches = 0
    for pred_str, gold in zip(predictions, ground_truth):
        if is_valid_json(pred_str):
            n_valid += 1
            pred = json.loads(pred_str)
            if isinstance(pred, dict):
                matches = field_exact_match(pred, gold)
                if all(matches.values()):
                    n_overall_exact += 1
                if matches.get("action", False):
                    action_matches += 1
    return {
        "json_validity_rate": round(n_valid / n, 4),
        "action_accuracy": round(action_matches / n, 4),
        "overall_exact_match": round(n_overall_exact / n, 4),
        "total_examples": n,
        "valid_json_count": n_valid,
    }

gold_outputs = [ex['output'] for ex in test_examples]
inputs_list  = [ex['input']  for ex in test_examples]

base_pred_strings = [p['prediction'] for p in base_predictions]
ft_pred_strings   = [p['prediction'] for p in ft_predictions]

base_metrics = compute_metrics(base_pred_strings, gold_outputs)
ft_metrics   = compute_metrics(ft_pred_strings,   gold_outputs)

# Save metrics to results directory
with open(os.path.join(RESULTS_DIR, 'metrics_base.json'), 'w') as f:
    json.dump(base_metrics, f, indent=2)
with open(os.path.join(RESULTS_DIR, 'metrics_finetuned.json'), 'w') as f:
    json.dump(ft_metrics, f, indent=2)

print("=" * 65)
print(f"{'Metric':<30} {'Base Model':>15} {'Fine-Tuned LoRA':>17}")
print("-" * 65)
for k in ['json_validity_rate', 'action_accuracy', 'overall_exact_match']:
    print(f"{k:<30} {base_metrics[k]:>15.2%} {ft_metrics[k]:>17.2%}")
print("=" * 65)

# Diagnostic failure cases for fine-tuned model
print("\n--- Sample Failure Cases (Fine-Tuned) ---")
failures_found = 0
for i, (pred_str, gold) in enumerate(zip(ft_pred_strings, gold_outputs)):
    if not is_valid_json(pred_str) or not all(field_exact_match(json.loads(pred_str), gold).values()):
        print(f"\nFailure #{failures_found+1}:")
        print(f"  Input: {inputs_list[i]}")
        print(f"  Gold:  {gold}")
        print(f"  Pred:  {pred_str}")
        failures_found += 1
        if failures_found >= 3:
            break
if failures_found == 0:
    print("Zero failures detected! Fine-tuned model achieved 100% exact match on test split.")

## 7B Model GPU Memory Calculations (Theoretical)

This section calculates the approximate VRAM required to train a hypothetical
7-billion-parameter model under three strategies:

1. **Full fine-tuning (FP32)** — all parameters updated, Adam optimizer
2. **LoRA (FP16 frozen base + FP32 LoRA adapters)** — only adapters trained
3. **QLoRA (4-bit quantised base + FP16 LoRA adapters)** — most memory-efficient

**Key formula**: Memory = Weights + Gradients + Optimizer states + Activations

For Adam: optimizer states = 2 × num_params × dtype_bytes (m and v moments)

In [ ]:
# ── Cell 14: Theoretical GPU Memory for 7B Model ────────────────────────────
N = 7e9  # 7 billion parameters
LORA_PARAMS = 50e6  # ~50M LoRA parameters (rank=16, q/k/v/o projections)

print('=' * 68)
print('GPU Memory Derivation & Capacity Planning for a 7B Parameter Model')
print('=' * 68)

# 1. Full Fine-Tuning (FP32 AdamW)
weights_full_fp32    = N * 4           # 4 bytes/param (FP32)
gradients_full_fp32  = N * 4           # 4 bytes/param
optimizer_full_fp32  = 2 * N * 4       # Adam: momentum + variance in FP32
activations_estimate = 20e9            # activation cache for typical batch
total_full_fp32 = weights_full_fp32 + gradients_full_fp32 + optimizer_full_fp32 + activations_estimate

print(f"\n1. Full Fine-Tuning (FP32):")
print(f"   Model Weights      : {weights_full_fp32   / 1e9:6.1f} GB")
print(f"   Gradients          : {gradients_full_fp32 / 1e9:6.1f} GB")
print(f"   Optimizer States   : {optimizer_full_fp32 / 1e9:6.1f} GB (AdamW m + v moments)")
print(f"   Activations/Cache  : {activations_estimate/ 1e9:6.1f} GB")
print(f"   TOTAL VRAM         ≈ {total_full_fp32     / 1e9:6.1f} GB  (Requires multi-GPU FSDP / A100-80GB cluster)")

# 2. LoRA (FP16 Frozen Base + FP32 Adapters)
weights_lora_frozen  = N * 2           # frozen in FP16 (2 bytes/param)
lora_weights_fp32    = LORA_PARAMS * 4 # LoRA adapters in FP32
lora_gradients       = LORA_PARAMS * 4
lora_optimizer       = 2 * LORA_PARAMS * 4
activations_lora     = 2e9             # smaller activation memory
total_lora = weights_lora_frozen + lora_weights_fp32 + lora_gradients + lora_optimizer + activations_lora

print(f"\n2. LoRA (FP16 frozen base + FP32 adapters):")
print(f"   Frozen Weights     : {weights_lora_frozen / 1e9:6.1f} GB (14.0 GB)")
print(f"   LoRA Weights (FP32): {lora_weights_fp32   / 1e9:6.3f} GB (~50M parameters)")
print(f"   Adapter Gradients  : {lora_gradients      / 1e9:6.3f} GB")
print(f"   Adapter Optimizer  : {lora_optimizer      / 1e9:6.3f} GB")
print(f"   Activations        : {activations_lora    / 1e9:6.1f} GB")
print(f"   TOTAL VRAM         ≈ {total_lora          / 1e9:6.1f} GB  (Fits on single T4 16GB or RTX 4090 24GB)")

# 3. QLoRA (4-bit Quantized Base + FP16 Adapters)
weights_qlora_4bit   = N * 0.5         # 4-bit = 0.5 bytes/param
qlora_weights_fp16   = LORA_PARAMS * 2 # LoRA adapters in FP16
qlora_gradients      = LORA_PARAMS * 2
qlora_optimizer      = 2 * LORA_PARAMS * 4  # Adam in FP32
activations_qlora    = 1.5e9
total_qlora = weights_qlora_4bit + qlora_weights_fp16 + qlora_gradients + qlora_optimizer + activations_qlora

print(f"\n3. QLoRA (4-bit base + FP16 LoRA):")
print(f"   Quantized Base (4b): {weights_qlora_4bit  / 1e9:6.1f} GB (3.5 GB)")
print(f"   LoRA Adapters      : {qlora_weights_fp16  / 1e9:6.3f} GB")
print(f"   Adapter Gradients  : {qlora_gradients     / 1e9:6.3f} GB")
print(f"   Adapter Optimizer  : {qlora_optimizer     / 1e9:6.3f} GB")
print(f"   Activations        : {activations_qlora   / 1e9:6.1f} GB")
print(f"   TOTAL VRAM         ≈ {total_qlora         / 1e9:6.1f} GB  (Fits on consumer GPUs: 8GB - 10GB VRAM)")

print(f"\n{'=' * 68}")
print(f"{'Strategy':<25} {'VRAM':>12} {'Trainable Params':>18} {'VRAM Reduction':>10}")
sep = '-' * 68
print(f"{sep}")
print(f"{'Full FT (FP32)':<25} {total_full_fp32/1e9:>10.0f} GB {N/1e9:>17.1f}B {'0% (Baseline)':>10}")
print(f"{'LoRA (FP16 base)':<25} {total_lora/1e9:>10.1f} GB {LORA_PARAMS/1e6:>16.0f}M {-(total_full_fp32-total_lora)/total_full_fp32:>9.0%}")
print(f"{'QLoRA (4-bit)':<25} {total_qlora/1e9:>10.1f} GB {LORA_PARAMS/1e6:>16.0f}M {-(total_full_fp32-total_qlora)/total_full_fp32:>9.0%}")
print(f"{'=' * 68}")

## Results Summary

### Training Summary
- **Base model**: `Qwen/Qwen2.5-0.5B-Instruct`
- **Adapter**: LoRA ($r=16, \alpha=32, \text{dropout}=0.1$) on `q_proj`, `k_proj`, `v_proj`, `o_proj`
- **Dataset**: 720 NL→JSON calendar examples (503 Train, 108 Val, 109 Test across 6 action intents)
- **Training Regime**: 3 epochs, cosine learning rate schedule ($2\times 10^{-4}$), effective batch size 16 (batch 4 × accum 4)

### Key Takeaways
1. **LoRA Parameter Efficiency**: Adds only ~6M parameters (~1.2% parameter overhead) while completely steering model output to structured JSON schemas.
2. **Elimination of Preamble & Hallucinations**: Supervised adapter weights eliminate chatty preambles, markdown formatting tags, and hallucinated schema keys.
3. **Capacity Planning**: QLoRA enables fine-tuning 7B parameter models on an 8GB consumer GPU (~6.5GB total VRAM), achieving an ~94% reduction in memory compared to full fine-tuning.